# 10 · 02 — Degradation, pre-registered

No model. The question is whether any channel *moves toward* a failure, which is what the
founding hypothesis claims and what three sessions have failed to find in power alone.

## Hypotheses, written before the first number

| id | claim | passes if |
|----|-------|-----------|
| **H1** | `mem_active` separates `OUT_OF_MEMORY` from `COMPLETED` — the spec's own mechanism | stratified AUC ≥ 0.65 at lead 0 |
| **H2** | `gpu_temp_max` separates hardware-kind failures (`FAILED`/`OOM`/`NODE_FAIL`) from `COMPLETED` | stratified AUC ≥ 0.60 at lead 0 |
| **H3** | whichever channel wins is **degradation**, not a level: it grows toward the event | AUC(lead 0) − AUC(lead 120) ≥ +0.05 |
| **H4** | the within-window **slope** carries signal | \|AUC − 0.5\| ≥ 0.05 for some channel |

## Controls, also pre-registered

* **C1 within-user.** Re-stratify by `user_id`. Session 08's entire power separation
  (AUC 0.727) evaporated here — it was workload identity, not degradation. Any channel
  that fails C1 is reporting who submitted the job.
* **C2 shuffle.** Labels permuted inside strata must return 0.50 ± 0.02.
* **C3 size.** Default strata are node-count × run-time decile, so neither big jobs nor
  long jobs can carry the result on their own.

## Power, stated before rather than after

May 2020 holds **21 `OUT_OF_MEMORY` and 2 `NODE_FAIL`** jobs. H1 is reported with its
confidence interval and should be read as a direction to check on six months, not as a
result. H2 rests on ~500 `FAILED`, which is enough to see an effect of the size the
hypothesis claims.

In [1]:
import json, os, sys
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("."))
import exalib as X

rng = np.random.default_rng(X.SEED)
os.makedirs("figures", exist_ok=True)
J = pd.read_parquet("data/jobs.parquet")
z = np.load("data/series.npz")
CH = [k[3:] for k in z.files if k.startswith("v__")]
V = {c: z[f"v__{c}"] for c in CH}
O = {c: z[f"o__{c}"] for c in CH}
K = np.diff(O[CH[0]])
print(f"{len(J):,} jobs, {len(CH)} channels, grid {X.GRID}s")

WIN        = 20                      # rows = minutes
LEADS      = [0, 10, 20, 30, 60, 120]
LEADS_SHORT= [0, 10, 20, 30]
NEED_FULL  = max(LEADS) + WIN        # 140 min of series
NEED_SHORT = max(LEADS_SHORT) + WIN  # 50 min
# gv100card0/3 are identically zero in 2020-05 (verified in 01) — the IPMI "GPU power"
# sensor carries nothing, so spec metric 6 stays unavailable and node_pow is the only
# power channel, exactly as in session 08.
TESTED = ["gpu_temp_max", "gpu0_mtemp", "mem_active", "node_pow", "swap_free",
          "proc_run", "cpu_wio", "load_one", "fan", "ambient"]

69,096 jobs, 22 channels, grid 60s


## Step 1 — strata and masks

The default stratum is (node count bucket) × (run-time decile): a separation must hold
*inside* jobs of comparable shape, so it cannot be "big jobs fail more".

In [2]:
sz = pd.cut(J.num_nodes_alloc, [0, 1, 2, 4, 8, 10**6], labels=False)
rt = pd.qcut(J.run_time, 10, labels=False, duplicates="drop")
STRATUM = (sz.astype(str) + "|" + rt.astype(str)).values
USER = J.user_id.values

is_comp = (J.job_state == "COMPLETED").values
is_hw   = J.job_state.isin(X.HARDWARE_STATES).values
is_oom  = (J.job_state == "OUT_OF_MEMORY").values
is_fail = (J.job_state == "FAILED").values
is_to   = (J.job_state == "TIMEOUT").values
for n, m in [("COMPLETED", is_comp), ("hardware-kind", is_hw), ("FAILED", is_fail),
             ("OOM", is_oom), ("TIMEOUT", is_to)]:
    print(f"  {n:14s} {m.sum():>6,}   with >={NEED_FULL}min series: {(m & (K >= NEED_FULL)).sum():>5,}")

  COMPLETED      52,967   with >=140min series: 19,491
  hardware-kind   8,224   with >=140min series: 3,179
  FAILED          7,663   with >=140min series: 2,970
  OOM               418   with >=140min series:   108
  TIMEOUT         7,905   with >=140min series: 4,302


## Step 2 — window extraction

`win(c, lead)` is the mean of channel `c` over the 20 minutes ending `lead` minutes
before the job ends. The cohort is fixed by `NEED` for the whole ladder, so every lead is
measured on the *same* jobs — otherwise a flat curve could just be a changing population.

In [3]:
def window(c, lead, need):
    v, o = V[c], O[c]
    out = np.full(len(J), np.nan, np.float64)
    sl  = np.full(len(J), np.nan, np.float64)
    for i in range(len(J)):
        if K[i] < need:
            continue
        end = o[i] + K[i] - lead
        seg = v[end - WIN:end]
        f = window_stats(seg)
        out[i], sl[i] = f[0], f[1]
    return out, sl

def window_stats(seg):
    s = seg[np.isfinite(seg)]
    if s.size == 0:
        return np.nan, np.nan
    if s.size == 1:
        return float(s[0]), 0.0
    x = np.arange(s.size, dtype=float)
    xm = x.mean(); den = ((x - xm) ** 2).sum()
    sl = float(((x - xm) * (s - s.mean())).sum() / den) if den > 0 else 0.0
    return float(s.mean()), sl

def boot_auc(x, pos, neg, strata, n=200):
    """Bootstrap over jobs — the unit of independence is the job, not the window."""
    idx = np.arange(len(x)); out = []
    for _ in range(n):
        b = rng.choice(idx, len(idx), replace=True)
        a = X.strat_auc(x[b], pos[b], neg[b], strata[b])
        if np.isfinite(a):
            out.append(a)
    return (np.nanpercentile(out, 2.5), np.nanpercentile(out, 97.5)) if out else (np.nan, np.nan)

## Step 3 — H1 and H2 at lead 0

In [4]:
rows = []
for c in TESTED:
    lvl, slp = window(c, 0, NEED_SHORT)
    for lbl, pos in [("hardware", is_hw), ("FAILED", is_fail),
                     ("OOM", is_oom), ("TIMEOUT", is_to)]:
        a = X.strat_auc(lvl, pos, is_comp, STRATUM)
        rows.append(dict(channel=c, mode=lbl, auc=a, n_pos=int((pos & np.isfinite(lvl)).sum()),
                         slope_auc=X.strat_auc(slp, pos, is_comp, STRATUM)))
L0 = pd.DataFrame(rows)
piv = L0.pivot(index="channel", columns="mode", values="auc")[["hardware", "FAILED", "OOM", "TIMEOUT"]]
print("stratified AUC vs COMPLETED, level over the last 20 min\n")
print(piv.round(3).sort_values("hardware", ascending=False).to_string())

stratified AUC vs COMPLETED, level over the last 20 min

mode          hardware  FAILED    OOM  TIMEOUT
channel                                       
mem_active       0.664   0.657  0.970    0.552
node_pow         0.609   0.609  0.627    0.398
proc_run         0.591   0.585  0.930    0.488
load_one         0.582   0.575  0.931    0.486
gpu_temp_max     0.574   0.575  0.561    0.424
fan              0.542   0.544  0.592    0.508
ambient          0.539   0.542  0.445    0.516
gpu0_mtemp       0.513   0.517  0.460    0.455
cpu_wio          0.502   0.501  0.548    0.499
swap_free        0.500   0.505  0.251    0.473


In [5]:
best_hw = piv.hardware.abs().sub(0.5).abs().idxmax()
lvl_hw, _ = window(best_hw, 0, NEED_SHORT)
ci_hw = boot_auc(lvl_hw, is_hw, is_comp, STRATUM)
lvl_mem, _ = window("mem_active", 0, NEED_SHORT)
a_oom = X.strat_auc(lvl_mem, is_oom, is_comp, STRATUM)
ci_oom = boot_auc(lvl_mem, is_oom, is_comp, STRATUM)
a_gpu = float(piv.loc["gpu_temp_max", "hardware"])
print(f"H1  mem_active vs OOM      AUC {a_oom:.3f}  95% CI [{ci_oom[0]:.3f}, {ci_oom[1]:.3f}]"
      f"  n_pos={int((is_oom & np.isfinite(lvl_mem)).sum())}"
      f"   -> {'PASS' if a_oom >= 0.65 else 'FAIL'} (threshold 0.65)")
print(f"H2  gpu_temp_max vs hw     AUC {a_gpu:.3f}"
      f"   -> {'PASS' if a_gpu >= 0.60 else 'FAIL'} (threshold 0.60)")
print(f"    strongest channel on hardware-kind: {best_hw} "
      f"AUC {piv.loc[best_hw,'hardware']:.3f} CI [{ci_hw[0]:.3f}, {ci_hw[1]:.3f}]")

H1  mem_active vs OOM      AUC 0.970  95% CI [0.914, 0.999]  n_pos=61   -> PASS (threshold 0.65)
H2  gpu_temp_max vs hw     AUC 0.574   -> FAIL (threshold 0.60)
    strongest channel on hardware-kind: mem_active AUC 0.664 CI [0.648, 0.680]


## Step 4 — H3: does it grow toward the event?

The same jobs, scored at six lead times. Flat means a level, not degradation.

In [6]:
lad = {}
for c in TESTED:
    r = {}
    for L in LEADS:
        lvl, _ = window(c, L, NEED_FULL)
        r[L] = X.strat_auc(lvl, is_hw, is_comp, STRATUM)
    lad[c] = r
LAD = pd.DataFrame(lad).T[LEADS]
LAD["growth"] = LAD[0] - LAD[120]
print(f"hardware-kind vs COMPLETED, cohort with >={NEED_FULL} min "
      f"(n_pos={(is_hw & (K >= NEED_FULL)).sum()}, n_neg={(is_comp & (K >= NEED_FULL)).sum()})\n")
print(LAD.round(3).sort_values("growth", key=abs, ascending=False).to_string())
gmax = LAD.growth.abs().max()
print(f"\nH3 largest |growth| = {gmax:.3f} -> {'PASS' if gmax >= 0.05 else 'FAIL'} (threshold 0.05)")

hardware-kind vs COMPLETED, cohort with >=140 min (n_pos=3179, n_neg=19491)

                  0     10     20     30     60    120  growth
mem_active    0.774  0.772  0.679  0.677  0.674  0.664   0.110
proc_run      0.674  0.677  0.616  0.609  0.608  0.594   0.081
load_one      0.672  0.676  0.616  0.612  0.608  0.594   0.078
node_pow      0.610  0.599  0.580  0.579  0.579  0.554   0.056
gpu_temp_max  0.570  0.561  0.546  0.546  0.549  0.528   0.041
gpu0_mtemp    0.506  0.496  0.485  0.485  0.486  0.468   0.037
swap_free     0.480  0.482  0.489  0.491  0.491  0.492  -0.012
ambient       0.546  0.546  0.542  0.537  0.537  0.538   0.008
fan           0.532  0.535  0.530  0.531  0.542  0.534  -0.003
cpu_wio       0.500  0.501  0.501  0.501  0.500  0.500  -0.000

H3 largest |growth| = 0.110 -> PASS (threshold 0.05)


## Step 5 — H4: slope

In [7]:
sl_rows = {}
for c in TESTED:
    _, slp = window(c, 0, NEED_SHORT)
    sl_rows[c] = {"hardware": X.strat_auc(slp, is_hw, is_comp, STRATUM),
                  "FAILED": X.strat_auc(slp, is_fail, is_comp, STRATUM),
                  "TIMEOUT": X.strat_auc(slp, is_to, is_comp, STRATUM)}
SL = pd.DataFrame(sl_rows).T
print("within-window slope, stratified AUC\n")
print(SL.round(3).to_string())
smax = (SL.hardware - 0.5).abs().max()
print(f"\nH4 max |slope AUC - 0.5| = {smax:.3f} -> {'PASS' if smax >= 0.05 else 'FAIL'}")

within-window slope, stratified AUC

              hardware  FAILED  TIMEOUT
gpu_temp_max     0.496   0.501    0.524
gpu0_mtemp       0.492   0.496    0.535
mem_active       0.697   0.693    0.576
node_pow         0.522   0.524    0.544
swap_free        0.490   0.495    0.492
proc_run         0.482   0.483    0.534
cpu_wio          0.501   0.501    0.499
load_one         0.500   0.501    0.531
fan              0.505   0.507    0.507
ambient          0.498   0.493    0.500

H4 max |slope AUC - 0.5| = 0.197 -> PASS


## Step 6 — C1 within-user, C2 shuffle, C3 size

C1 is the control that decided session 08. If a separation is really degradation it must
survive comparing a user's failing jobs against that same user's completed jobs.

In [8]:
ctl = []
for c in TESTED:
    lvl, _ = window(c, 0, NEED_SHORT)
    ok = np.isfinite(lvl)
    nu = sum(1 for u in np.unique(USER[ok])
             if (is_hw & ok & (USER == u)).sum() >= 5 and (is_comp & ok & (USER == u)).sum() >= 5)
    pool = is_hw | is_comp                            # C2: permute the label inside strata,
    sh = np.zeros(len(J), bool)                       # among the jobs that carry one
    for s in np.unique(STRATUM):
        m = (STRATUM == s) & pool
        if m.any():
            sh[m] = rng.permutation(is_hw[m])
    ctl.append(dict(channel=c,
                    default=X.strat_auc(lvl, is_hw, is_comp, STRATUM),
                    within_user=X.strat_auc(lvl, is_hw, is_comp, USER),
                    shuffled=X.strat_auc(lvl, sh, pool & ~sh, STRATUM),
                    n_users=nu))
C = pd.DataFrame(ctl).set_index("channel")
C["survives_C1"] = (C.within_user - 0.5).abs() >= 0.5 * (C.default - 0.5).abs()
print(C.round(3).to_string())

              default  within_user  shuffled  n_users  survives_C1
channel                                                           
gpu_temp_max    0.574        0.504     0.508       93        False
gpu0_mtemp      0.513        0.503     0.506       93        False
mem_active      0.664        0.609     0.496       58         True
node_pow        0.609        0.492     0.502       93        False
swap_free       0.500        0.497     0.499       58         True
proc_run        0.591        0.492     0.498       58        False
cpu_wio         0.502        0.500     0.500       58        False
load_one        0.582        0.474     0.507       58        False
fan             0.542        0.497     0.489       97        False
ambient         0.539        0.503     0.498       97        False


## Step 7 — figure

In [9]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for c in LAD.index:
    ax[0].plot(LEADS, LAD.loc[c, LEADS].values, marker="o", ms=3, label=c)
ax[0].axhline(0.5, color="k", lw=.8, ls="--")
ax[0].invert_xaxis(); ax[0].set_xlabel("minutes before job end"); ax[0].set_ylabel("stratified AUC")
ax[0].set_title(f"H3 · hardware-kind vs COMPLETED\n(same {(is_hw & (K>=NEED_FULL)).sum()} failing jobs at every lead)")
ax[0].legend(fontsize=6, ncol=2)
d = C.reset_index().sort_values("default", key=lambda s: (s - .5).abs(), ascending=True)
yy = np.arange(len(d))
ax[1].barh(yy + .2, d.default - .5, .38, label="size×runtime strata")
ax[1].barh(yy - .2, d.within_user - .5, .38, label="within user")
ax[1].set_yticks(yy); ax[1].set_yticklabels(d.channel, fontsize=8)
ax[1].axvline(0, color="k", lw=.8); ax[1].set_xlabel("stratified AUC − 0.5")
ax[1].set_title("C1 · what survives holding the user fixed"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig("figures/02_degradation.png", dpi=130)
print("wrote figures/02_degradation.png")

wrote figures/02_degradation.png


In [10]:
out = dict(H1=dict(auc=float(a_oom), ci=[float(ci_oom[0]), float(ci_oom[1])],
                   n_pos=int((is_oom & np.isfinite(lvl_mem)).sum()),
                   verdict="PASS" if a_oom >= 0.65 else "FAIL"),
           H2=dict(auc=float(a_gpu), verdict="PASS" if a_gpu >= 0.60 else "FAIL"),
           H3=dict(max_growth=float(gmax), verdict="PASS" if gmax >= 0.05 else "FAIL"),
           H4=dict(max_slope=float(smax), verdict="PASS" if smax >= 0.05 else "FAIL"),
           best_hardware_channel=best_hw,
           level_auc=piv.round(4).to_dict(), ladder=LAD.round(4).to_dict(),
           controls=C.round(4).drop(columns=["survives_C1"]).to_dict())
json.dump(out, open("data/degradation.json", "w"), indent=1)
print(json.dumps({k: out[k] for k in ["H1", "H2", "H3", "H4"]}, indent=1))

{
 "H1": {
  "auc": 0.9702999702999703,
  "ci": [
   0.9136384742002772,
   0.9989858966606286
  ],
  "n_pos": 61,
  "verdict": "PASS"
 },
 "H2": {
  "auc": 0.5736533582810243,
  "verdict": "FAIL"
 },
 "H3": {
  "max_growth": 0.11037869360528596,
  "verdict": "PASS"
 },
 "H4": {
  "max_slope": 0.1965203612687153,
  "verdict": "PASS"
 }
}


## Step 8 — the reframing: normalise each job against itself

Everything above compares **levels between jobs**, and a level cannot separate "this job
is degrading" from "this job is large". That is why `mem_active` scores 0.664 on a level
and falls to 0.609 within a user: the level is partly the user's habitual footprint.

The hypothesis says a job *deteriorates*. So subtract each job's own first 20 minutes and
ask whether the **change** separates. This removes the level and the submitter's baseline
in one step, and is the strictest reading of "degradação gradual".

Reported on `FAILED` alone — `OUT_OF_MEMORY` is excluded so the OOM jobs (61 positives on
six months), which are separable on the level anyway, cannot carry it.


In [11]:
NEED_N = 160                                   # 120 lead + 20 window + 20 baseline
okn = K >= NEED_N
BASE = {}
def wmean(c, i, a, b):
    s = V[c][O[c][i] + a:O[c][i] + b]; s = s[np.isfinite(s)]
    return s.mean() if s.size else np.nan

NORM_CH = ["mem_active", "node_pow", "gpu_temp_max", "load_one", "proc_run"]
for c in NORM_CH:
    BASE[c] = np.array([wmean(c, i, 0, 20) if okn[i] else np.nan for i in range(len(J))])

nu_f = sum(1 for u in np.unique(USER[okn])
           if (is_fail & okn & (USER == u)).sum() >= 5 and (is_comp & okn & (USER == u)).sum() >= 5)
print(f"cohort >={NEED_N} min: FAILED {int((is_fail & okn).sum())}, "
      f"OOM {int((is_oom & okn).sum())}, COMPLETED {int((is_comp & okn).sum())}")
print(f"users with >=5 FAILED and >=5 COMPLETED: {nu_f}   "
      f"(the within-user control is underpowered at one month)\n")

rows = []
for c in NORM_CH:
    for L in LEADS[::-1]:
        cur = np.array([wmean(c, i, K[i] - L - WIN, K[i] - L) if okn[i] else np.nan
                        for i in range(len(J))])
        d = cur - BASE[c]
        rows.append(dict(channel=c, lead=L,
                         auc=X.strat_auc(d, is_fail, is_comp, STRATUM),
                         within_user=X.strat_auc(d, is_fail, is_comp, USER),
                         med_fail=np.nanmedian(d[is_fail & okn]),
                         med_comp=np.nanmedian(d[is_comp & okn])))
N = pd.DataFrame(rows)
print("within-job normalised (window at lead) − (own first 20 min), FAILED vs COMPLETED\n")
print(N.pivot(index="lead", columns="channel", values="auc").round(3).to_string())
print("\nsame statistic, stratified by user instead:\n")
print(N.pivot(index="lead", columns="channel", values="within_user").round(3).to_string())

cohort >=160 min: FAILED 2718, OOM 103, COMPLETED 18393
users with >=5 FAILED and >=5 COMPLETED: 59   (the within-user control is underpowered at one month)



within-job normalised (window at lead) − (own first 20 min), FAILED vs COMPLETED

channel  gpu_temp_max  load_one  mem_active  node_pow  proc_run
lead                                                           
0               0.498     0.622       0.792     0.494     0.526
10              0.489     0.635       0.791     0.486     0.531
20              0.487     0.639       0.763     0.475     0.541
30              0.486     0.640       0.753     0.467     0.534
60              0.492     0.650       0.769     0.475     0.564
120             0.459     0.637       0.762     0.433     0.529

same statistic, stratified by user instead:

channel  gpu_temp_max  load_one  mem_active  node_pow  proc_run
lead                                                           
0               0.463     0.482       0.613     0.475     0.450
10              0.466     0.481       0.609     0.473     0.465
20              0.472     0.476       0.605     0.479     0.456
30              0.471     0.458       0.

In [12]:
mem = N[N.channel == "mem_active"].set_index("lead")
print("mem_active growth over the job's own baseline, GB\n")
print(pd.DataFrame({"AUC": mem.auc.round(3),
                    "FAILED": (mem.med_fail / 1e6).round(2),
                    "COMPLETED": (mem.med_comp / 1e6).round(2)}).to_string())

pool = is_fail | is_comp
sh = np.zeros(len(J), bool)
for s in np.unique(STRATUM):
    m = (STRATUM == s) & pool
    if m.any():
        sh[m] = rng.permutation(is_fail[m])
cur = np.array([wmean("mem_active", i, K[i] - 140, K[i] - 120) if okn[i] else np.nan
                for i in range(len(J))])
d120 = cur - BASE["mem_active"]
print(f"\nC2 shuffle on the same statistic at lead 120: "
      f"{X.strat_auc(d120, sh, pool & ~sh, STRATUM):.3f}  (want 0.50)")

mem_active growth over the job's own baseline, GB

        AUC  FAILED  COMPLETED
lead                          
120   0.762    5.19       0.48
60    0.769    6.59       0.47
30    0.753    6.87       0.45
20    0.763    6.51       0.44
10    0.791    6.02       0.37
0     0.792    5.85       0.29



C2 shuffle on the same statistic at lead 120: 0.494  (want 0.50)


**This is the only result in ten sessions that matches the founding hypothesis.** A job
that will fail grows its memory footprint by **5–7 GB** over its run; a job that completes
grows **0.3–0.5 GB** and stays flat. Within-user AUC holds at **0.567–0.613** at every lead
on **58 qualifying users**, and the shuffle control reads 0.494.

**Read the monotonicity with care — `11_onset/01` re-examined it and it does not survive.**
Two findings there bear on this cell directly:

* bootstrapped over jobs, the CIs at lead 120 and lead 0 **overlap heavily**, so the rise
  0.567 → 0.613 is not a resolved effect, only an ordering of point estimates;
* re-indexed by **elapsed time since the job started** — which is the only prediction time a
  scheduler can actually choose — the same statistic on the same jobs is **flat at ~0.59 from
  minute 40 onward**.

The separation is real. What is not supported is that it *grows toward the failure*: it is a
level established early and held, the same shape `06_headroom` found for `OUT_OF_MEMORY`.
Operationally that is the better outcome, but it means a pre-failure horizon has nothing to
mark, which is why session 11 exists.


In [13]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for c in NORM_CH:
    s = N[N.channel == c].sort_values("lead")
    ax[0].plot(s.lead, s.auc, marker="o", ms=4, label=c)
ax[0].axhline(0.5, color="k", lw=.8, ls="--"); ax[0].invert_xaxis()
ax[0].set_xlabel("minutes before job end"); ax[0].set_ylabel("stratified AUC")
ax[0].set_title(f"normalised to each job's own first 20 min\nFAILED (n={int((is_fail&okn).sum())}) "
                f"vs COMPLETED (n={int((is_comp&okn).sum())})")
ax[0].legend(fontsize=7)
s = N[N.channel == "mem_active"].sort_values("lead")
ax[1].plot(s.lead, s.med_fail / 1e6, marker="o", label="FAILED")
ax[1].plot(s.lead, s.med_comp / 1e6, marker="o", label="COMPLETED")
ax[1].invert_xaxis(); ax[1].set_xlabel("minutes before job end")
ax[1].set_ylabel("GB grown since the job's own start")
ax[1].set_title("mem_active · the trajectory itself"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures/02_normalised.png", dpi=130)
N.to_csv("data/normalised.csv", index=False)
print("wrote figures/02_normalised.png")

wrote figures/02_normalised.png
